# Workflow de derramamento de óleo com OpenOil (ensemble)

Este notebook executa um workflow completo de previsão de deriva de óleo com o **OpenOil** (módulo de óleo do [OpenDrift](https://opendrift.github.io/)):

1. **Configuração** do cenário e sorteio dos membros do ensemble
2. **Ensemble**: N simulações independentes, executadas em paralelo (uma tarefa = um processo)
3. **Perfil de execução** de cada tarefa (duração, CPU, pico de memória, tamanho da saída), exportado em CSV para alimentar um simulador de workflows
4. **Agregação**: mapa de probabilidade de presença de óleo, tempo de chegada à costa e balanço de massa
5. **Animação** de um membro

**Cenário padrão:** vazamento de 300 m³ de óleo cru médio durante 6 h, a cerca de 30 km da costa de Maricá/Niterói (RJ), simulado por 72 h.

> **Atenção: as forçantes são sintéticas.** Vento e corrente são constantes no tempo e no espaço, sorteados por membro entre dois regimes (frente fria de sul e vento nordeste). A linha de costa e a física do óleo são reais, mas o resultado **não é uma previsão**. A última seção explica como trocar por dados reais.

## 1. Instalação
Leva de 1 a 2 minutos no Colab.

In [ ]:
%pip install -q opendrift

## 2. A tarefa do workflow: um membro do ensemble
O script abaixo é gravado em disco e chamado como processo separado, do mesmo jeito que seria executado dentro de um contêiner.

In [ ]:
%%writefile membro.py
"""Uma tarefa do workflow: simula UM membro do ensemble com o OpenOil.

Uso:  python membro.py '<json com os parametros>'
Gera: <saida>.nc (trajetorias) e <saida>.json (perfil de execucao da tarefa).
"""
import json, os, resource, sys, time
from datetime import datetime, timedelta

VARIAVEIS = ['lon', 'lat', 'z', 'status', 'mass_oil', 'mass_evaporated',
             'mass_dispersed', 'water_fraction']


def simular(p):
    inicio = time.time()
    from opendrift.models.openoil import OpenOil
    o = OpenOil(loglevel=50)

    # --- Forcantes -------------------------------------------------------
    # Se houver arquivos NetCDF reais (correntes/vento), eles tem prioridade;
    # caso contrario usa-se o campo constante sorteado para este membro.
    arquivos = [a for a in (p.get('arquivo_correntes'), p.get('arquivo_vento')) if a]
    if arquivos:
        from opendrift.readers import reader_netCDF_CF_generic
        o.add_reader([reader_netCDF_CF_generic.Reader(a) for a in arquivos])
    if not p.get('arquivo_vento'):
        o.set_config('environment:constant:x_wind', p['vento_u'])
        o.set_config('environment:constant:y_wind', p['vento_v'])
    if not p.get('arquivo_correntes'):
        o.set_config('environment:constant:x_sea_water_velocity', p['corrente_u'])
        o.set_config('environment:constant:y_sea_water_velocity', p['corrente_v'])
    o.set_config('environment:constant:sea_water_temperature', p['temperatura_agua'])

    # --- Processos fisicos ----------------------------------------------
    o.set_config('processes:evaporation', True)
    o.set_config('processes:emulsification', True)
    o.set_config('processes:dispersion', True)
    o.set_config('drift:vertical_mixing', True)
    o.set_config('drift:wind_uncertainty', 1.0)       # m/s, turbulencia nao resolvida
    o.set_config('drift:current_uncertainty', 0.05)   # m/s
    o.set_config('general:coastline_action', 'stranding')

    # --- Vazamento -------------------------------------------------------
    t0 = datetime.fromisoformat(p['inicio'])
    o.seed_elements(lon=p['lon'], lat=p['lat'], radius=p['raio_m'],
                    number=p['particulas'],
                    time=[t0, t0 + timedelta(hours=p['horas_vazamento'])],
                    z=0, m3_per_hour=p['m3_por_hora'], oil_type=p['oleo'])

    o.run(duration=timedelta(hours=p['horas_simuladas']),
          time_step=p['passo_s'], time_step_output=3600,
          outfile=p['saida'] + '.nc', export_variables=VARIAVEIS)

    perfil = {
        'membro': p['membro'],
        'inicio_epoch': inicio,
        'fim_epoch': time.time(),
        'duracao_s': round(time.time() - inicio, 2),
        'cpu_s': round(sum(os.times()[:2]), 2),
        'pico_memoria_mb': round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024, 1),
        'saida_mb': round(os.path.getsize(p['saida'] + '.nc') / 1e6, 2),
    }
    with open(p['saida'] + '.json', 'w') as f:
        json.dump(perfil, f)
    return perfil


if __name__ == '__main__':
    print(json.dumps(simular(json.loads(sys.argv[1]))))

## 3. Configuração do cenário e do ensemble
Ajuste aqui. `N_MEMBROS` e `PARTICULAS` controlam o custo; o Colab gratuito tem 2 núcleos e cada tarefa usa cerca de 2,6 GB de memória, então mantenha `WORKERS = 2`.

In [ ]:
import json, math, random

# --- Custo computacional ---
N_MEMBROS  = 16      # largura do fan-out
PARTICULAS = 1000    # partículas por membro
WORKERS    = 2       # tarefas simultâneas
SEMENTE    = 42

# --- Cenário ---
CENARIO = dict(
    inicio='2026-11-20T06:00:00',
    lon=-42.90, lat=-23.25,          # ponto do vazamento
    raio_m=1500,
    horas_vazamento=16, m3_por_hora=250,   # 300 m3 no total
    oleo='GENERIC MEDIUM CRUDE',
    horas_simuladas=172, passo_s=900,
    temperatura_agua=30.0,
    particulas=PARTICULAS,
    arquivo_correntes=None,          # caminho de NetCDF real (opcional)
    arquivo_vento=None,              # caminho de NetCDF real (opcional)
)

def sortear_membro(i, rng):
    # Dois regimes de vento: frente fria (vento de sul) e nordeste predominante
    if rng.random() < 0.5:
        regime, vel, de_onde = 'frente fria', rng.gauss(9, 2), rng.gauss(190, 25)
    else:
        regime, vel, de_onde = 'nordeste', rng.gauss(7, 2), rng.gauss(50, 20)
    vel = max(vel, 1.0)
    # Corrente para sudoeste (sentido da Corrente do Brasil)
    c_vel, c_para = max(rng.gauss(0.20, 0.07), 0.0), rng.gauss(240, 20)
    r = math.radians
    return dict(CENARIO, membro=i, regime=regime, saida=f'membro_{i:03d}',
                vento_vel=round(vel, 2), vento_de=round(de_onde % 360, 1),
                vento_u=-vel * math.sin(r(de_onde)), vento_v=-vel * math.cos(r(de_onde)),
                corrente_u=c_vel * math.sin(r(c_para)), corrente_v=c_vel * math.cos(r(c_para)))

rng = random.Random(SEMENTE)
membros = [sortear_membro(i, rng) for i in range(N_MEMBROS)]
for m in membros:
    print(f"membro {m['membro']:02d}  {m['regime']:<11}  vento {m['vento_vel']:5.1f} m/s de {m['vento_de']:5.1f} graus")

## 4. Execução do ensemble
Cada membro roda em um processo próprio. Com os valores padrão leva cerca de 4 minutos.

In [ ]:
import subprocess, sys, time
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd

def executar(p):
    r = subprocess.run([sys.executable, 'membro.py', json.dumps(p)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"membro {p['membro']} falhou:\n{r.stderr[-2000:]}")
    return json.loads(r.stdout.strip().splitlines()[-1])

t_ini = time.time()
perfis = []
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    futuros = [pool.submit(executar, p) for p in membros]
    for f in as_completed(futuros):
        perfis.append(f.result())
        print(f"[{time.time() - t_ini:6.1f} s] membro {perfis[-1]['membro']:02d} concluído "
              f"({perfis[-1]['duracao_s']:.1f} s, {perfis[-1]['pico_memoria_mb']:.0f} MB)")
makespan = time.time() - t_ini

perfil = pd.DataFrame(perfis).sort_values('membro').reset_index(drop=True)
perfil['atividade'] = 'simulacao'
perfil['inicio_s'] = perfil['inicio_epoch'] - t_ini
perfil['fim_s'] = perfil['fim_epoch'] - t_ini
print(f"\nMakespan: {makespan:.1f} s | soma das tarefas: {perfil.duracao_s.sum():.1f} s | "
      f"speedup: {perfil.duracao_s.sum() / makespan:.2f}x com {WORKERS} workers")

## 5. Agregação do ensemble
Junta as saídas de todos os membros em produtos de decisão. Esta etapa também é medida como uma tarefa do workflow.

In [ ]:
import numpy as np, xarray as xr, os, resource

t_agg = time.time()
RES = 0.02   # resolução da grade, em graus (~2 km)

def preencher(a):
    # Repete o último valor válido de cada partícula após ela ser desativada
    idx = np.where(~np.isnan(a), np.arange(a.shape[1]), 0)
    np.maximum.accumulate(idx, axis=1, out=idx)
    return np.take_along_axis(a, idx, axis=1)

dados = []
for m in membros:
    ds = xr.open_dataset(m['saida'] + '.nc').load()
    lon, lat, status = ds.lon.values, ds.lat.values, ds.status.values
    encalhou = np.nan_to_num(preencher(status), nan=0) == 1          # (partícula, tempo)
    massa = {v: np.nan_to_num(preencher(ds[v].values), nan=0.0)
             for v in ('mass_oil', 'mass_evaporated', 'mass_dispersed')}
    horas = (ds.time.values - ds.time.values[0]) / np.timedelta64(1, 'h')
    t_costa = horas[encalhou.any(axis=0)].min() if encalhou.any() else np.nan
    dados.append(dict(
        membro=m['membro'], regime=m['regime'], lon=lon, lat=lat, horas=horas,
        lon_f=preencher(lon), lat_f=preencher(lat), encalhou=encalhou,
        t_costa=t_costa, frac_costa=encalhou[:, -1].mean(),
        no_mar=(massa['mass_oil'] * ~encalhou).sum(0), na_costa=(massa['mass_oil'] * encalhou).sum(0),
        evaporado=massa['mass_evaporated'].sum(0), disperso=massa['mass_dispersed'].sum(0)))
    ds.close()

# Grade comum a todos os membros
todos_lon = np.concatenate([d['lon'][~np.isnan(d['lon'])] for d in dados])
todos_lat = np.concatenate([d['lat'][~np.isnan(d['lat'])] for d in dados])
x0 = min(todos_lon.min(), -43.45) - 0.1; x1 = max(todos_lon.max(), -42.40) + 0.1
y0 = min(todos_lat.min(), -23.45) - 0.1; y1 = max(todos_lat.max(), -22.70) + 0.1
bx, by = np.arange(x0, x1 + RES, RES), np.arange(y0, y1 + RES, RES)

# Probabilidade = fração dos membros em que o óleo passou por cada célula
atingiu = np.zeros((len(by) - 1, len(bx) - 1))
for d in dados:
    ok = ~np.isnan(d['lon'])
    h, _, _ = np.histogram2d(d['lat'][ok], d['lon'][ok], bins=[by, bx])
    atingiu += h > 0
prob = atingiu / len(dados)

resumo = pd.DataFrame([{k: d[k] for k in ('membro', 'regime', 't_costa', 'frac_costa')} for d in dados])
n_costa = resumo.t_costa.notna().sum()
print(f"Membros em que o óleo toca a costa: {n_costa} de {len(dados)} ({100 * n_costa / len(dados):.0f}%)")
if n_costa:
    print(f"Tempo até o primeiro toque: mínimo {resumo.t_costa.min():.0f} h, mediana {resumo.t_costa.median():.0f} h")
print(resumo.groupby('regime').agg(membros=('membro', 'count'), tocam_costa=('t_costa', 'count'),
                                    fracao_media_na_costa=('frac_costa', 'mean')).round(2))

# Registra a agregação como tarefa e exporta o perfil do workflow
perfil = pd.concat([perfil, pd.DataFrame([dict(
    membro=-1, atividade='agregacao', duracao_s=round(time.time() - t_agg, 2),
    inicio_s=t_agg - t_ini, fim_s=time.time() - t_ini,
    pico_memoria_mb=round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024, 1))])],
    ignore_index=True)
colunas = ['atividade', 'membro', 'inicio_s', 'fim_s', 'duracao_s', 'cpu_s', 'pico_memoria_mb', 'saida_mb']
perfil[colunas].round(2).to_csv('perfil_tarefas.csv', index=False)
perfil[colunas].round(2)

## 6. Mapa de probabilidade

In [ ]:
import matplotlib.pyplot as plt
from roaring_landmask import RoaringLandmask

# Máscara de terra real (GSHHG), a mesma que o OpenDrift usa para o encalhe
gx, gy = np.arange(x0, x1, 0.004), np.arange(y0, y1, 0.004)
GX, GY = np.meshgrid(gx, gy)
terra = RoaringLandmask.new().contains_many(GX.ravel().astype('float64'),
                                            GY.ravel().astype('float64')).reshape(GX.shape)

def desenhar_base(ax):
    ax.set_facecolor('#eaf2f8')
    ax.contourf(GX, GY, terra, levels=[0.5, 1.5], colors=['#d9d2c5'])
    ax.contour(GX, GY, terra, levels=[0.5], colors='#7a7265', linewidths=0.6)
    for nome, lo, la in [('Niterói', -43.07, -22.89), ('Rio de Janeiro', -43.42, -22.90), ('Maricá', -42.82, -22.90)]:
        if x0 < lo < x1 and y0 < la < y1:
            ax.text(lo, la, nome, fontsize=9, ha='center', va='bottom', color='#3b352c')
    ax.plot(CENARIO['lon'], CENARIO['lat'], marker='*', ms=16, color='black', mec='white', ls='', label='Ponto do vazamento')
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
    ax.set_aspect(1 / np.cos(np.radians((y0 + y1) / 2)))
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')

fig, ax = plt.subplots(figsize=(11, 7))
desenhar_base(ax)
pm = ax.pcolormesh(bx, by, np.ma.masked_equal(prob, 0) * 100, cmap='YlOrRd', vmin=0, vmax=100)
fig.colorbar(pm, ax=ax, shrink=0.75, label='Membros do ensemble em que o óleo passou pela célula (%)')
ax.legend(loc='lower right')
ax.set_title(f"Probabilidade de presença de óleo em {CENARIO['horas_simuladas']} h "
             f"({len(dados)} membros, forçantes sintéticas)")
fig.savefig('mapa_probabilidade.png', dpi=150, bbox_inches='tight'); plt.show()

## 7. Destino do óleo, chegada à costa e execução das tarefas

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 4.8))

# (a) Balanço de massa médio do ensemble
h = dados[0]['horas']
partes = ['no_mar', 'na_costa', 'evaporado', 'disperso']
medias = [np.mean([d[k] for d in dados], axis=0) / 1000 for k in partes]
axs[0].stackplot(h, medias, labels=['No mar', 'Na costa', 'Evaporado', 'Disperso na coluna d\'água'],
                 colors=['#1f2933', '#b5651d', '#9fb8c8', '#4f8fba'])
axs[0].set(xlabel='Horas desde o início do vazamento', ylabel='Massa de óleo (t)',
           title='Destino do óleo (média do ensemble)', xlim=(0, h[-1]))
axs[0].legend(loc='upper left', fontsize=8)

# (b) Tempo até o primeiro toque na costa
tc = resumo.dropna(subset=['t_costa'])
if len(tc):
    axs[1].barh([f"{int(m):02d}" for m in tc.sort_values('t_costa').membro], tc.sort_values('t_costa').t_costa, color='#b5651d')
    axs[1].invert_yaxis()
axs[1].set(xlabel='Horas até o primeiro toque na costa', ylabel='Membro',
           title=f'Chegada à costa ({len(tc)} de {len(dados)} membros)')

# (c) Gantt das tarefas
sim = perfil[perfil.atividade == 'simulacao'].sort_values('inicio_s')
axs[2].barh([f"{int(m):02d}" for m in sim.membro], sim.duracao_s, left=sim.inicio_s, color='#4f8fba', label='Simulação')
ag = perfil[perfil.atividade == 'agregacao'].iloc[0]
axs[2].barh(['agreg.'], [ag.duracao_s], left=[ag.inicio_s], color='#1f2933', label='Agregação')
axs[2].invert_yaxis()
axs[2].set(xlabel='Tempo de parede (s)', ylabel='Tarefa', title=f'Execução do workflow ({WORKERS} workers)')
axs[2].legend(fontsize=8)
plt.tight_layout(); fig.savefig('resumo_workflow.png', dpi=150, bbox_inches='tight'); plt.show()

## 8. Animação de um membro
Mostra o membro com maior fração de óleo na costa (ou o membro 0, se nenhum tocar a costa).

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

d = max(dados, key=lambda d: d['frac_costa'])
fig, ax = plt.subplots(figsize=(9, 6))
desenhar_base(ax)
mar = ax.scatter([], [], s=4, color='#1f2933', label='Óleo no mar')
costa = ax.scatter([], [], s=6, color='#d1495b', label='Óleo na costa')
ax.legend(loc='lower right')

def quadro(k):
    ativo = ~np.isnan(d['lon'][:, k]) & ~d['encalhou'][:, k]
    enc = d['encalhou'][:, k]
    mar.set_offsets(np.c_[d['lon'][ativo, k], d['lat'][ativo, k]])
    costa.set_offsets(np.c_[d['lon_f'][enc, k], d['lat_f'][enc, k]])
    ax.set_title(f"Membro {d['membro']:02d} ({d['regime']}) | {d['horas'][k]:.0f} h após o início")

anim = FuncAnimation(fig, quadro, frames=range(0, len(d['horas']), 2), interval=150)
plt.close(fig)
HTML(anim.to_jshtml())

## Próximos passos

**Usar forçantes reais.** Baixe correntes e vento em NetCDF no padrão CF (por exemplo, Copernicus Marine para correntes e ERA5 ou GFS para vento), envie os arquivos para o Colab e preencha `arquivo_correntes` e `arquivo_vento` em `CENARIO`. O script passa a usar esses arquivos no lugar dos campos constantes. Nesse caso, a incerteza do ensemble deve vir de outras fontes: instante e posição do vazamento, volume, tipo de óleo ou previsões meteorológicas diferentes. Este caminho não foi testado neste notebook.

**Alimentar o simulador de workflows.** O arquivo `perfil_tarefas.csv` traz, por tarefa, início, fim, duração, tempo de CPU, pico de memória e tamanho da saída. Varie `PARTICULAS` e `horas_simuladas` para obter perfis de tarefas mais leves e mais pesadas.

**Conteinerizar.** `membro.py` recebe os parâmetros em JSON e grava dois arquivos; basta uma imagem com `pip install opendrift` para executá-lo como atividade de um workflow.